# Lab 3: The Chain Rule — Manual Backpropagation

## Setup

In [ ]:
!pip install -q -r requirements.txt

## Introduction

### KEY CONCEPT — The Chain Rule

If `f` depends on `e`, and `e` depends on `a`, then:

```
df/da = (df/de) × (de/da)
```

This "chain" of multiplications traces how **ANY** input affects the final output, no matter how deeply nested the computation is.

### ANALOGY — Car vs. Bicycle vs. Walking

- Walking speed: 1 km/h
- Cycling: 10× faster than walking → 10 km/h
- Driving: 10× faster than cycling → 100 km/h

```
Car speed = (cycling/walking) × (driving/cycling) × walking = 10 × 10 × 1 = 100
```

The chain rule multiplies these "speedups" (sensitivities) together.

### In Neural Networks

**"Gradient" means "sensitivity"**:

If `a.grad = 10`, then nudging `a` by +0.001 changes the output by ≈ +0.010.

## Setup: Value Class (Pre-implemented)

For this lab, we provide a complete forward-only Value class:

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        self.data  = float(data)
        self.grad  = 0.0
        self._prev = set(_children)
        self._op   = _op
        self.label = label

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), '+')

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), '*')

    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __neg__(self):         return self * -1
    def __sub__(self, other):  return self + (-other)

    def __repr__(self):
        lbl = f"'{self.label}' " if self.label else ''
        return f"Value({lbl}data={self.data:.4f}, grad={self.grad:.4f})"

## Visualizing the Computation Graph

Before we start, let's see how to visualize computation graphs. This helps you understand the flow of gradients!

In [ ]:
from utils import visualize_graph

# Example: visualize a simple expression
x = Value(2.0, label='x')
y = Value(3.0, label='y')
z = x * y; z.label = 'z'

# Draw the computation graph
visualize_graph(z, title="Simple Computation Graph: z = x * y")

## Demo: Manual Backprop on `f = (a + b) * c`

### Forward Pass

```python
e = a + b  =  2 + (-3)  =  -1
f = e * c  =  (-1) * 10 =  -10
```

### Backward Pass (work right to left)

```python
df/df = 1                                 (by definition)
df/de = c.data  =  10                     (d(e*c)/de = c)
df/dc = e.data  =  -1                     (d(e*c)/dc = e)
df/da = df/de × de/da  =  10 × 1  =  10  (chain rule; de/da = 1 since e = a+b)
df/db = df/de × de/db  =  10 × 1  =  10  (chain rule; de/db = 1)
```

In [ ]:
print("=" * 55)
print("DEMO: f = (a + b) * c")
print("=" * 55)

a = Value(2.0,  label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')

e = a + b;  e.label = 'e'
f = e * c;  f.label = 'f'

# Set gradients manually:
f.grad = 1.0
c.grad = e.data           # df/dc = e
e.grad = c.data           # df/de = c
a.grad = e.grad * 1.0     # chain rule: df/da = df/de × de/da = 10 × 1
b.grad = e.grad * 1.0     # chain rule: df/db = df/de × de/db = 10 × 1

print(f"  a.grad = {a.grad}   (expected  10.0)")
print(f"  b.grad = {b.grad}   (expected  10.0)")
print(f"  c.grad = {c.grad}   (expected  -1.0)")

# Sanity-check with a numerical gradient (bump a by h and see how f changes):
h = 1e-4
f_plus = (Value(a.data + h) + Value(b.data)) * Value(c.data)
numerical_a = (f_plus.data - f.data) / h
print(f"\n  Numerical df/da ≈ {numerical_a:.4f}  (should match a.grad = {a.grad})")

### Visualize the Computation Graph

Let's draw the graph to see the structure and gradients:

In [ ]:
from utils import visualize_graph

# Visualize the computation graph with gradients
visualize_graph(f, title="Manual Backprop: f = (a + b) * c")

# You should see:
# - Nodes showing both data and grad values
# - Gradients: f.grad=1.0, e.grad=10.0, c.grad=-1.0, a.grad=10.0, b.grad=10.0
# - Arrows showing gradient flows backwards from f to inputs

## Exercise: Manual Backprop on `L = ((a * b) + c) * d`

### Problem Setup

Given: `a=2`, `b=3`, `c=1`, `d=-1`

### Step 1: Build the Expression

Build using Value objects:
```python
P = a * b
Q = P + c
L = Q * d
```

**Verify**: `L.data` should be `-7.0`

### Step 2: Set Gradients (Work Backwards)

Start from `L.grad = 1.0` and work backwards:

**HINTS**:
- For `L = Q * d`: `dL/dQ = d.data`, `dL/dd = Q.data`
- For `Q = P + c`: `dQ/dP = 1`, `dQ/dc = 1`
- For `P = a * b`: `dP/da = b.data`, `dP/db = a.data`
- Chain rule: `a.grad = (dL/dQ) × (dQ/dP) × (dP/da)`

### Your Task

Fill in the code below:

In [ ]:
print("=" * 55)
print("EXERCISE: L = ((a * b) + c) * d")
print("=" * 55)

a = Value(2.0,  label='a')
b = Value(3.0,  label='b')
c = Value(1.0,  label='c')
d = Value(-1.0, label='d')

# TODO: Build the expression
# P = a * b;  P.label = 'P'
# Q = P + c;  Q.label = 'Q'
# L = Q * d;  L.label = 'L'
# print(f"L.data = {L.data}   (expected -7.0)")

# TODO: Set gradients manually, working backwards from L
# L.grad = 1.0
# d.grad = ...
# Q.grad = ...
# c.grad = ...
# P.grad = ...
# b.grad = ...
# a.grad = ...

print("\n⚠️  Fill in the TODOs above to see gradients")

### Visualize Your Solution

After completing the exercise, visualize the graph to validate your gradient flow:

In [ ]:
# Uncomment after filling in the exercise:
# from utils import visualize_graph
# visualize_graph(L, title="Your Solution: L = ((a * b) + c) * d")

# Check that:
# ✓ All nodes have non-zero grad values
# ✓ Gradient flows from L back to a, b, c, d
# ✓ Intermediate nodes (P, Q) have correct gradients
# ✓ Compare with the expected values in the solution

### Correct Answer

<details>
<summary>Click to reveal solution</summary>

```python
# Build the expression
P = a * b;  P.label = 'P'  # P = 2 * 3 = 6
Q = P + c;  Q.label = 'Q'  # Q = 6 + 1 = 7
L = Q * d;  L.label = 'L'  # L = 7 * (-1) = -7

# Set gradients manually
L.grad = 1.0              # dL/dL = 1
d.grad = Q.data           # dL/dd = Q = 7
Q.grad = d.data           # dL/dQ = d = -1
c.grad = Q.grad * 1.0     # dL/dc = dL/dQ × dQ/dc = -1 × 1 = -1
P.grad = Q.grad * 1.0     # dL/dP = dL/dQ × dQ/dP = -1 × 1 = -1
b.grad = P.grad * a.data  # dL/db = dL/dP × dP/db = -1 × 2 = -2
a.grad = P.grad * b.data  # dL/da = dL/dP × dP/da = -1 × 3 = -3
```

**Expected gradients**:
- `a.grad = -3.0`
- `b.grad = -2.0`
- `c.grad = -1.0`
- `d.grad =  7.0`
</details>

### Numerical Gradient Checker

After filling in your gradients, run this to verify:

In [ ]:
def numerical_grad(val, h=1e-4):
    """Estimate dL/d(val) by rebuilding L after bumping val.data by ±h."""
    original = val.data

    val.data = original + h
    _a, _b, _c, _d = Value(a.data), Value(b.data), Value(c.data), Value(d.data)
    L_plus = ((_a * _b) + _c) * _d

    val.data = original - h
    _a, _b, _c, _d = Value(a.data), Value(b.data), Value(c.data), Value(d.data)
    L_minus = ((_a * _b) + _c) * _d

    val.data = original
    return (L_plus.data - L_minus.data) / (2 * h)

# Uncomment after filling in gradients:
# print(f"\n  a: analytical = {a.grad:.4f}  |  numerical = {numerical_grad(a):.4f}")
# print(f"  b: analytical = {b.grad:.4f}  |  numerical = {numerical_grad(b):.4f}")
# print(f"  c: analytical = {c.grad:.4f}  |  numerical = {numerical_grad(c):.4f}")
# print(f"  d: analytical = {d.grad:.4f}  |  numerical = {numerical_grad(d):.4f}")
# print("\n🎉 If analytical matches numerical, you got it right!")

## Edge Cases to Test

Try these additional expressions:

In [ ]:
# Edge case 1: Same variable used twice
x = Value(3.0, label='x')
y = x + x  # x appears twice
y.grad = 1.0
# When x appears twice, we need to sum both contributions:
x.grad = 1.0 + 1.0  # dy/dx from left + dy/dx from right
print(f"Edge case 1: x used twice → x.grad = {x.grad} (expected 2.0)")

# Edge case 2: Longer chain
a = Value(2.0)
b = Value(3.0)
c = Value(4.0)
d = a * b  # d = 6
e = d + c  # e = 10
f = e * e  # f = 100
f.grad = 1.0
e.grad = 2 * e.data  # df/de = 2*e (since f = e²)
c.grad = e.grad * 1.0  # chain rule
d.grad = e.grad * 1.0
b.grad = d.grad * a.data
a.grad = d.grad * b.data
print(f"Edge case 2: a.grad = {a.grad}, b.grad = {b.grad} (via long chain)")

print("\n✅ Edge cases help you understand gradient flow!")

## Summary

You've learned:
- **The chain rule**: multiply sensitivities along a path
- **Manual backpropagation**: work backwards from output to inputs
- **Numerical verification**: always check your analytical gradients!
- **Multivariate case**: when a variable is used multiple times, sum all contributions

Next: Lab 4 will automate this entire process!